# Phase 1: Figure 생성 및 결과 정리 (논문용)

## 포함 내용
1. Classification 결과 Table & Bar Chart
2. 2D Embedding Visualization (t-SNE, UMAP)
3. Confusion Matrix + Frobenius Norm
4. F1 Score 등 추가 평가 지표
5. Ablation Study (Six-pack barcode 기여도 분석)

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.base import clone
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['axes.unicode_minus'] = False

# UMAP (설치 필요할 수 있음)
try:
    import umap
    HAS_UMAP = True
except:
    !pip install umap-learn -q
    import umap
    HAS_UMAP = True

print('All imports loaded.')

## 2. Ground Truth & Adjacent Phases

In [ ]:
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1,M2,M3,M4,M5,M6,M7,M8])

def get_label_from_index(task_id):
    idx = task_id - 1
    return GROUND_TRUTH_M[(idx%64)//8][idx//64][idx%8]

def extract_adjacent_phases(matrices):
    adj = set()
    for M in matrices:
        M = np.array(M)
        r,c = M.shape
        for i in range(r):
            for j in range(c):
                cur = M[i,j]
                for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                    ni,nj = i+di,j+dj
                    if 0<=ni<r and 0<=nj<c and cur!=M[ni,nj]:
                        adj.add(tuple(sorted([int(cur),int(M[ni,nj])])))
    d = {}
    for p1,p2 in adj:
        d.setdefault(p1,[]).append(p2)
        d.setdefault(p2,[]).append(p1)
    return d

ADJACENT_PHASES = extract_adjacent_phases(GROUND_TRUTH_M)

def soft_accuracy_score(y_true, y_pred, adj=ADJACENT_PHASES):
    n = len(y_true)
    correct = sum(1 for t,p in zip(y_true,y_pred)
                  if t==p or (t in adj and p in adj[t]) or (p in adj and t in adj[p]))
    return correct/n if n>0 else 0.0

ROMAN = {0:'i',1:'ii',2:'iii',3:'iv',4:'v',6:'vi',7:'vii',9:'ix',10:'x',11:'xi',12:'xii',13:'xiii'}
CLASSES = sorted(np.unique(GROUND_TRUTH_M))
CLASS_LABELS = [ROMAN.get(c,str(c)) for c in CLASSES]
print(f'{len(CLASSES)} classes: {CLASS_LABELS}')

## 3. 데이터 로딩

In [ ]:
VECTOR_DIR = '/content/drive/MyDrive/URP/1224_Vectors'

def load_generic_data(data_dir, prefix):
    pattern = os.path.join(data_dir, f'{prefix}_*.npz')
    npz_files = sorted(glob.glob(pattern))
    X_list, y_list = [], []
    for fp in npz_files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            arr_0 = data['arr_0'].item() if hasattr(data['arr_0'],'item') and data['arr_0'].ndim==0 else (data['arr_0'][0] if data['arr_0'].shape==(1,) else data['arr_0'])
            arr_1 = data['arr_1'].item() if hasattr(data['arr_1'],'item') and data['arr_1'].ndim==0 else (data['arr_1'][0] if data['arr_1'].shape==(1,) else data['arr_1'])
            feats = []
            for arr in [arr_0, arr_1]:
                if isinstance(arr, dict):
                    for k in sorted(arr.keys()):
                        v = arr[k]
                        if isinstance(v, dict):
                            for dk in sorted(v.keys()): feats.extend(np.asarray(v[dk]).flatten())
                        else: feats.extend(np.asarray(v).flatten())
                else: feats.extend(np.asarray(arr).flatten())
            X_list.append(feats); y_list.append(label)
        except Exception as e: print(f'  Error {fp}: {e}')
    if not X_list: return None, None
    return np.nan_to_num(np.array(X_list),nan=0.,posinf=0.,neginf=0.), np.array(y_list)

def extract_statistical_features(barcode):
    if len(barcode)==0: return np.zeros(12)
    bc=np.array(barcode)
    if bc.ndim==1:
        if len(bc)%2==0: bc=bc.reshape(-1,2)
        else: bc=bc[:len(bc)//2*2].reshape(-1,2) if len(bc)>2 else np.array([[0.,0.]])
    if bc.ndim==1 or bc.shape[1]<2: return np.zeros(12)
    ls=bc[:,1]-bc[:,0]; b=bc[:,0]; d=bc[:,1]
    feats=[len(bc),np.mean(ls),np.std(ls),np.max(ls),np.min(ls),np.sum(ls),
           np.mean(b),np.std(b),np.mean(d),np.std(d),np.median(ls)]
    p=ls/np.sum(ls) if np.sum(ls)>0 else ls; p=p[p>0]
    feats.append(-np.sum(p*np.log(p+1e-10)) if len(p)>0 else 0)
    return np.array(feats)

BARCODE_TYPES = ['domain','codomain','relative','image','kernel','cokernel']

def load_sixpack_rips_ablation(data_dir, selected_types=None):
    """Sixpack_Rips 로드. selected_types로 특정 barcode만 선택 가능 (ablation용)."""
    if selected_types is None: selected_types = BARCODE_TYPES
    npz_files = sorted(glob.glob(os.path.join(data_dir,'Sixpack_Rips_*.npz')))
    X_list, y_list = [], []
    for fp in npz_files:
        try:
            sim_idx = int(os.path.basename(fp).split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(fp, allow_pickle=True)
            sp = {'A_to_B':data['arr_0'].item(),'B_to_A':data['arr_1'].item()}
            feats = []
            for d in ['A_to_B','B_to_A']:
                dd = sp[d]
                for bt in BARCODE_TYPES:
                    for dim_key in [0,1]:
                        if bt in selected_types and bt in dd and dim_key in dd[bt]:
                            feats.extend(extract_statistical_features(np.array(dd[bt][dim_key])))
                        elif bt in selected_types:
                            feats.extend(np.zeros(12))
            X_list.append(feats); y_list.append(label)
        except Exception as e: print(f'  Error {fp}: {e}')
    if not X_list: return None, None
    return np.nan_to_num(np.array(X_list),nan=0.,posinf=0.,neginf=0.), np.array(y_list)

print('Data loading functions defined.')

In [ ]:
# 모든 데이터셋 로드
datasets = {}
METHODS = {
    'Inter_PI': ('Inter_PI','Inter_PI'),
    '3D_PI': ('3D_PI','3D_PI'),
    'Ord_PI': ('Ord_PI','Ord_PI'),
    'Sixpack_Chroma': ('Sixpack_Chroma','Sixpack_Chroma'),
}

for name,(dirn,prefix) in METHODS.items():
    print(f'Loading {name}...',end=' ')
    X,y = load_generic_data(os.path.join(VECTOR_DIR,dirn), prefix)
    if X is not None:
        datasets[name] = {'X':X,'y':y}
        print(f'✓ {X.shape}')
    else: print('✗')

# Sixpack_Rips (별도 로더)
print('Loading Sixpack_Rips...',end=' ')
X,y = load_sixpack_rips_ablation(os.path.join(VECTOR_DIR,'Sixpack_Rips'))
if X is not None:
    datasets['Sixpack_Rips'] = {'X':X,'y':y}
    print(f'✓ {X.shape}')

# 조합 데이터셋
if 'Inter_PI' in datasets and 'Ord_PI' in datasets:
    datasets['Inter+Ord'] = {
        'X': np.hstack([datasets['Inter_PI']['X'], datasets['Ord_PI']['X']]),
        'y': datasets['Inter_PI']['y']}
    print(f'Inter+Ord: {datasets["Inter+Ord"]["X"].shape}')
if '3D_PI' in datasets and 'Ord_PI' in datasets:
    datasets['3D_Ord'] = {
        'X': np.hstack([datasets['3D_PI']['X'], datasets['Ord_PI']['X']]),
        'y': datasets['3D_PI']['y']}
    print(f'3D_Ord: {datasets["3D_Ord"]["X"].shape}')

print(f'\nTotal: {len(datasets)} datasets loaded.')

## 4. 통합 평가 함수

In [ ]:
PCA_DIM = 20
N_SPLITS = 5

def full_evaluate(X, y, pca_dim=PCA_DIM, n_splits=N_SPLITS):
    """모든 분류기 + 모든 지표 한번에 평가."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    if pca_dim and Xs.shape[1]>pca_dim:
        Xs = PCA(n_components=pca_dim,random_state=42).fit_transform(Xs)

    clfs = {
        'KNN(k=3)':KNeighborsClassifier(3),
        'KNN(k=12)':KNeighborsClassifier(12),
        'SVM-RBF':SVC(kernel='rbf',C=1.,gamma='scale',random_state=42),
        'SVM-Linear':SVC(kernel='linear',C=1.,random_state=42),
        'RF':RandomForestClassifier(100,random_state=42),
    }
    skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=42)
    results = {}
    all_y_true, all_y_pred = {}, {}  # for confusion matrix

    for cn,ct in clfs.items():
        metrics = {'soft':[],'strict':[],'f1':[],'precision':[],'recall':[]}
        yt_all, yp_all = [], []
        for tri,tei in skf.split(Xs,y):
            c = clone(ct); c.fit(Xs[tri],y[tri]); yp = c.predict(Xs[tei])
            metrics['soft'].append(soft_accuracy_score(y[tei],yp))
            metrics['strict'].append(accuracy_score(y[tei],yp))
            metrics['f1'].append(f1_score(y[tei],yp,average='weighted',zero_division=0))
            metrics['precision'].append(precision_score(y[tei],yp,average='weighted',zero_division=0))
            metrics['recall'].append(recall_score(y[tei],yp,average='weighted',zero_division=0))
            yt_all.extend(y[tei]); yp_all.extend(yp)
        results[cn] = {k:{'mean':np.mean(v)*100,'std':np.std(v)*100} for k,v in metrics.items()}
        all_y_true[cn] = np.array(yt_all)
        all_y_pred[cn] = np.array(yp_all)

    return results, Xs, all_y_true, all_y_pred

print('Evaluation function defined.')

## 5. 전체 데이터셋 평가

In [ ]:
all_results = {}
all_embeddings = {}  # for 2D viz
all_preds = {}  # for confusion matrix

for name, data in datasets.items():
    print(f'\n[{name}] ({data["X"].shape})')
    res, Xs, yt, yp = full_evaluate(data['X'], data['y'])
    all_results[name] = res
    all_embeddings[name] = {'X_scaled': Xs, 'y': data['y']}
    all_preds[name] = {'y_true': yt, 'y_pred': yp}
    # Print best
    best = max(res, key=lambda k: res[k]['soft']['mean'])
    r = res[best]
    print(f'  Best: {best} → Soft={r["soft"]["mean"]:.2f}% | Strict={r["strict"]["mean"]:.2f}% | F1={r["f1"]["mean"]:.2f}%')

print('\n✓ All evaluations complete.')

## 6. [1-1] Classification 결과 Table & Bar Chart

실험 조건별로 분류 결과를 정리합니다.

In [ ]:
# 6-1. 종합 결과 테이블
rows = []
for method, res in all_results.items():
    for clf, metrics in res.items():
        rows.append({
            'Method': method,
            'Dim': datasets[method]['X'].shape[1],
            'Classifier': clf,
            'Soft Acc (%)': f"{metrics['soft']['mean']:.2f}±{metrics['soft']['std']:.2f}",
            'Strict Acc (%)': f"{metrics['strict']['mean']:.2f}±{metrics['strict']['std']:.2f}",
            'F1 (%)': f"{metrics['f1']['mean']:.2f}±{metrics['f1']['std']:.2f}",
            'Precision (%)': f"{metrics['precision']['mean']:.2f}±{metrics['precision']['std']:.2f}",
            'Recall (%)': f"{metrics['recall']['mean']:.2f}±{metrics['recall']['std']:.2f}",
            '_soft': metrics['soft']['mean'],
        })

df_full = pd.DataFrame(rows)

# Best per method
df_best = df_full.sort_values('_soft',ascending=False).groupby('Method').first().reset_index()
df_best = df_best.sort_values('_soft',ascending=False)
print('=== Best Classifier per Method ===')
print(df_best[['Method','Dim','Classifier','Soft Acc (%)','Strict Acc (%)','F1 (%)']].to_string(index=False))

# CSV 저장
df_full.drop(columns=['_soft']).to_csv('/content/drive/MyDrive/URP/phase1_full_results.csv',index=False)
print('\nCSV saved.')

In [ ]:
# 6-2. Bar Chart - 모든 방법 × SVM-Linear 비교
TARGET_CLF = 'SVM-Linear'
methods_order = sorted(all_results.keys(),
    key=lambda m: all_results[m].get(TARGET_CLF,{}).get('soft',{}).get('mean',0))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax_i, (metric, label) in enumerate([('soft','Soft Accuracy'),('strict','Strict Accuracy'),('f1','F1 Score')]):
    vals = [all_results[m][TARGET_CLF][metric]['mean'] for m in methods_order]
    stds = [all_results[m][TARGET_CLF][metric]['std'] for m in methods_order]
    colors = ['#FF5722' if 'Sixpack_Rips' in m else '#2196F3' for m in methods_order]
    bars = axes[ax_i].barh(range(len(methods_order)), vals, xerr=stds,
        color=colors, alpha=0.85, capsize=3)
    axes[ax_i].set_yticks(range(len(methods_order)))
    axes[ax_i].set_yticklabels(methods_order, fontsize=10)
    axes[ax_i].set_xlabel(f'{label} (%)')
    axes[ax_i].set_title(f'{label} ({TARGET_CLF})')
    axes[ax_i].set_xlim([0,105])
    axes[ax_i].grid(axis='x',alpha=0.3)
    for bar,val in zip(bars,vals):
        axes[ax_i].text(val+1, bar.get_y()+bar.get_height()/2, f'{val:.1f}',
            va='center',fontsize=9)

plt.suptitle(f'Classification Results ({TARGET_CLF}, PCA {PCA_DIM}D, {N_SPLITS}-fold CV)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/fig_classification_barchart.png',dpi=200,bbox_inches='tight')
plt.show()

## 7. [1-2] 2D Embedding Visualization

t-SNE와 UMAP을 사용한 2D 시각화.

In [ ]:
# 2D embedding for top methods
VIZ_METHODS = ['Sixpack_Rips','Ord_PI','Inter_PI','Sixpack_Chroma']
VIZ_METHODS = [m for m in VIZ_METHODS if m in all_embeddings]

# Color map for classes
cmap = plt.cm.get_cmap('tab20', len(CLASSES))
class_colors = {c: cmap(i) for i,c in enumerate(CLASSES)}

fig, axes = plt.subplots(len(VIZ_METHODS), 2, figsize=(16, 5*len(VIZ_METHODS)))
if len(VIZ_METHODS)==1: axes = axes.reshape(1,-1)

for row, method in enumerate(VIZ_METHODS):
    Xs = all_embeddings[method]['X_scaled']
    y = all_embeddings[method]['y']

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    emb_tsne = tsne.fit_transform(Xs)
    ax = axes[row, 0]
    for c in CLASSES:
        mask = y==c
        ax.scatter(emb_tsne[mask,0], emb_tsne[mask,1], c=[class_colors[c]],
            label=ROMAN.get(c,str(c)), s=15, alpha=0.7)
    ax.set_title(f'{method} — t-SNE')
    ax.set_xticks([]); ax.set_yticks([])
    if row==0: ax.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=7, ncol=2)

    # UMAP
    reducer = umap.UMAP(n_components=2, random_state=42)
    emb_umap = reducer.fit_transform(Xs)
    ax = axes[row, 1]
    for c in CLASSES:
        mask = y==c
        ax.scatter(emb_umap[mask,0], emb_umap[mask,1], c=[class_colors[c]],
            label=ROMAN.get(c,str(c)), s=15, alpha=0.7)
    ax.set_title(f'{method} — UMAP')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/fig_2d_embedding.png',dpi=200,bbox_inches='tight')
plt.show()

## 8. [1-3] Confusion Matrix + Frobenius Norm

In [ ]:
# 주요 방법들의 confusion matrix
CM_METHODS = ['Sixpack_Rips','Inter_PI','Ord_PI','Sixpack_Chroma']
CM_METHODS = [m for m in CM_METHODS if m in all_preds]
CM_CLF = 'SVM-Linear'

n_methods = len(CM_METHODS)
fig, axes = plt.subplots(2, n_methods, figsize=(5*n_methods, 10))
if n_methods==1: axes = axes.reshape(-1,1)

frob_results = {}

for col, method in enumerate(CM_METHODS):
    yt = all_preds[method]['y_true'][CM_CLF]
    yp = all_preds[method]['y_pred'][CM_CLF]
    cm = confusion_matrix(yt, yp, labels=CLASSES)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:,np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)

    # Frobenius distance from identity
    identity = np.eye(len(CLASSES))
    frob = np.linalg.norm(cm_norm - identity, 'fro')
    frob_results[method] = frob

    # Raw counts
    im = axes[0,col].imshow(cm, cmap='Blues', interpolation='nearest')
    axes[0,col].set_title(f'{method}\n(counts)', fontsize=10)
    axes[0,col].set_xticks(range(len(CLASS_LABELS)))
    axes[0,col].set_xticklabels(CLASS_LABELS, rotation=45, fontsize=7)
    axes[0,col].set_yticks(range(len(CLASS_LABELS)))
    axes[0,col].set_yticklabels(CLASS_LABELS, fontsize=7)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            c = 'white' if cm[i,j]>cm.max()/2 else 'black'
            axes[0,col].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=6,color=c)

    # Normalized
    im = axes[1,col].imshow(cm_norm, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
    axes[1,col].set_title(f'{method}\nFrob={frob:.3f}', fontsize=10)
    axes[1,col].set_xticks(range(len(CLASS_LABELS)))
    axes[1,col].set_xticklabels(CLASS_LABELS, rotation=45, fontsize=7)
    axes[1,col].set_yticks(range(len(CLASS_LABELS)))
    axes[1,col].set_yticklabels(CLASS_LABELS, fontsize=7)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            c = 'white' if cm_norm[i,j]>0.5 else 'black'
            axes[1,col].text(j,i,f'{cm_norm[i,j]:.2f}',ha='center',va='center',fontsize=5,color=c)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/fig_confusion_matrices.png',dpi=200,bbox_inches='tight')
plt.show()

print('\nFrobenius Distance from Identity (lower=better):')
for m,f in sorted(frob_results.items(), key=lambda x:x[1]):
    print(f'  {m:20s}: {f:.4f}')

## 9. [1-4] F1 Score & 추가 평가 지표

In [ ]:
# Per-class F1 Score for best method
BEST_METHOD = 'Sixpack_Rips'
if BEST_METHOD in all_preds:
    yt = all_preds[BEST_METHOD]['y_true'][CM_CLF]
    yp = all_preds[BEST_METHOD]['y_pred'][CM_CLF]
    print(f'=== Classification Report: {BEST_METHOD} ({CM_CLF}) ===')
    print(classification_report(yt, yp, target_names=CLASS_LABELS, zero_division=0))

# Multi-metric comparison table
print('\n=== All Methods × All Metrics (SVM-Linear) ===')
metric_rows = []
for method in sorted(all_results.keys()):
    r = all_results[method].get('SVM-Linear',{})
    if r:
        metric_rows.append({
            'Method': method,
            'Soft Acc': f"{r['soft']['mean']:.2f}",
            'Strict Acc': f"{r['strict']['mean']:.2f}",
            'F1': f"{r['f1']['mean']:.2f}",
            'Precision': f"{r['precision']['mean']:.2f}",
            'Recall': f"{r['recall']['mean']:.2f}",
        })
print(pd.DataFrame(metric_rows).to_string(index=False))

## 10. [1-5] Ablation Study

Six-pack의 6개 barcode 중 어떤 것이 분류 정확도에 기여하는지 분석.
- `domain`, `codomain`: Ordinary PI와 동일한 정보
- `image`, `kernel`, `cokernel`: Six-pack 고유 정보
- `relative`: 상대 호몰로지 정보

In [ ]:
# Ablation: 각 barcode type 조합별 정확도
RIPS_DIR = os.path.join(VECTOR_DIR, 'Sixpack_Rips')

ablation_configs = {
    'Full (all 6)': BARCODE_TYPES,
    'domain+codomain only': ['domain','codomain'],
    'image only': ['image'],
    'kernel only': ['kernel'],
    'cokernel only': ['cokernel'],
    'relative only': ['relative'],
    'image+kernel+cokernel': ['image','kernel','cokernel'],
    'domain+codomain+image': ['domain','codomain','image'],
    'domain+codomain+kernel': ['domain','codomain','kernel'],
    'domain+codomain+cokernel': ['domain','codomain','cokernel'],
    'domain+codomain+relative': ['domain','codomain','relative'],
    'w/o image': [t for t in BARCODE_TYPES if t!='image'],
    'w/o kernel': [t for t in BARCODE_TYPES if t!='kernel'],
    'w/o cokernel': [t for t in BARCODE_TYPES if t!='cokernel'],
    'w/o relative': [t for t in BARCODE_TYPES if t!='relative'],
}

ablation_results = {}
print('Ablation Study - Sixpack_Rips')
print('='*80)

for config_name, selected in ablation_configs.items():
    X, y = load_sixpack_rips_ablation(RIPS_DIR, selected_types=selected)
    if X is None: continue
    res, _, _, _ = full_evaluate(X, y, pca_dim=None)  # No PCA (288D or less)
    best_clf = max(res, key=lambda k: res[k]['soft']['mean'])
    r = res[best_clf]
    # Also get SVM-Linear specifically
    r_lin = res.get('SVM-Linear',r)
    ablation_results[config_name] = {
        'dim': X.shape[1],
        'types': selected,
        'best_clf': best_clf,
        'best_soft': r['soft']['mean'],
        'best_strict': r['strict']['mean'],
        'linear_soft': r_lin['soft']['mean'],
        'linear_strict': r_lin['strict']['mean'],
    }
    print(f'  {config_name:35s} (dim={X.shape[1]:4d}) | '
          f'Soft={r_lin["soft"]["mean"]:.2f}% | Strict={r_lin["strict"]["mean"]:.2f}%')
    del X, y; gc.collect()

print('='*80)

In [ ]:
# Ablation 시각화
configs = list(ablation_results.keys())
soft_vals = [ablation_results[c]['linear_soft'] for c in configs]
strict_vals = [ablation_results[c]['linear_strict'] for c in configs]

# Sort by soft acc
order = np.argsort(soft_vals)
configs_sorted = [configs[i] for i in order]
soft_sorted = [soft_vals[i] for i in order]
strict_sorted = [strict_vals[i] for i in order]

fig, ax = plt.subplots(figsize=(12, max(6, len(configs)*0.4)))
y_pos = np.arange(len(configs_sorted))

colors = []
for c in configs_sorted:
    if c == 'Full (all 6)': colors.append('#FF5722')
    elif c == 'domain+codomain only': colors.append('#4CAF50')
    elif 'w/o' in c: colors.append('#FFC107')
    else: colors.append('#2196F3')

ax.barh(y_pos+0.15, soft_sorted, 0.3, color=colors, alpha=0.9, label='Soft Acc')
ax.barh(y_pos-0.15, strict_sorted, 0.3, color=[c+'80' if not c.startswith('#FF') else '#FF572280' for c in colors], alpha=0.7, label='Strict Acc')

ax.set_yticks(y_pos)
ax.set_yticklabels(configs_sorted, fontsize=9)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Ablation Study: Barcode Type Contribution (SVM-Linear, Sixpack_Rips)')
ax.legend()
ax.set_xlim([0,105])
ax.grid(axis='x',alpha=0.3)

for val,yp in zip(soft_sorted,y_pos):
    ax.text(val+0.5, yp+0.15, f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/URP/fig_ablation_study.png',dpi=200,bbox_inches='tight')
plt.show()

# Ablation 요약 테이블
print('\n=== Ablation Study Summary ===')
abl_rows = []
for c in sorted(ablation_results.keys(), key=lambda k: ablation_results[k]['linear_soft'], reverse=True):
    r = ablation_results[c]
    delta = r['linear_soft'] - ablation_results['domain+codomain only']['linear_soft']
    abl_rows.append({'Config':c, 'Dim':r['dim'],
        'Soft':f"{r['linear_soft']:.2f}", 'Strict':f"{r['linear_strict']:.2f}",
        'Δ vs Ord':f"{delta:+.2f}"})
print(pd.DataFrame(abl_rows).to_string(index=False))